In [1]:
import os
import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Digite sua GROQ_API_KEY: ")
print("✅ GROQ_API_KEY configurada!")

✅ GROQ_API_KEY configurada!


In [2]:
# Importações necessárias
import os
from dataclasses import dataclass
from dotenv import load_dotenv
from pydantic_ai import Agent, RunContext

# Carregar variáveis de ambiente
load_dotenv()

# Simulação de sistema de pedidos para teste
class FakeOrderSystem:
    def get_order_status(self, order_id: str) -> str:
        return "Em processamento" if order_id == "123" else "Pedido não encontrado."

# Definição do tipo de dependência
@dataclass
class TestDeps:
    order_system: FakeOrderSystem

# Criar um agente com a ferramenta
agent = Agent(
    'groq:openai/gpt-oss-120b',
    deps_type=TestDeps,
    system_prompt="Você é um assistente que ajuda a verificar o status de pedidos."
)

# Definição da ferramenta que será testada
@agent.tool()
async def get_order_status(ctx, order_id: str) -> str:
    """
    Consulta o status de um pedido no sistema.
    
    Args:
        ctx: Contexto com acesso ao sistema de pedidos
        order_id: ID do pedido a ser consultado
        
    Returns:
        Status atual do pedido
    """
    return ctx.deps.order_system.get_order_status(order_id)

# Função para testar através do agente
async def testar_via_agente():
    # Criar as dependências
    fake_deps = TestDeps(order_system=FakeOrderSystem())
    
    # Testar com ID válido
    result_valido = await agent.run(
        "Qual é o status do pedido 123?", 
        deps=fake_deps
    )
    
    # Testar com ID inválido
    result_invalido = await agent.run(
        "Qual é o status do pedido 456?", 
        deps=fake_deps
    )
    
    print("Resposta para pedido válido:")
    print(result_valido.output)
    print("\nResposta para pedido inválido:")
    print(result_invalido.output)
    
    # Verificação manual dos resultados
    if "Em processamento" in result_valido.output:
        print("\n✅ Teste para pedido válido passou")
    else:
        print("\n❌ Teste para pedido válido falhou")
        
    if "não encontrado" in result_invalido.output.lower():
        print("✅ Teste para pedido inválido passou")
    else:
        print("❌ Teste para pedido inválido falhou")
    
    return result_valido, result_invalido

# Executar teste via agente
await testar_via_agente()

Resposta para pedido válido:
O pedido **123** está **em processamento** no momento. Se precisar de mais informações ou quiser acompanhar a evolução, é só me avisar!

Resposta para pedido inválido:
Desculpe, não encontrei nenhum pedido com o número **456** no sistema. Por favor, verifique se o número está correto ou forneça mais detalhes para que eu possa ajudar melhor.

❌ Teste para pedido válido falhou
❌ Teste para pedido inválido falhou


(AgentRunResult(output='O pedido **123** está **em processamento** no momento. Se precisar de mais informações ou quiser acompanhar a evolução, é só me avisar!'),
 AgentRunResult(output='Desculpe, não encontrei nenhum pedido com o número **456** no sistema. Por favor, verifique se o número está correto ou forneça mais detalhes para que eu possa ajudar melhor.'))

In [3]:
from pydantic_ai import Agent
from dotenv import load_dotenv
from typing import Dict, Optional, Any
import time

# Carregar variáveis de ambiente
load_dotenv()

# Classe para simular o monitoramento de uso
class UsageMonitor:
    def __init__(self, 
                 request_limit: Optional[int] = 50,
                 tokens_per_request_limit: Optional[int] = 4000,
                 tokens_per_minute_limit: Optional[int] = 20000):
        self.request_limit = request_limit
        self.tokens_per_request_limit = tokens_per_request_limit
        self.tokens_per_minute_limit = tokens_per_minute_limit
        
        # Contadores
        self.request_count = 0
        self.total_tokens = 0
        self.minute_tokens = 0
        self.minute_start = time.time()
        self.usage_log: Dict[str, Any] = {}
    
    def check_limits(self, estimated_tokens: int = 0) -> bool:
        # Atualizar contagem por minuto se necessário
        current_time = time.time()
        if current_time - self.minute_start > 60:
            self.minute_tokens = 0
            self.minute_start = current_time
            
        # Verificar limites
        if self.request_limit and self.request_count >= self.request_limit:
            raise Exception(f"Limite de requisições excedido: {self.request_count}/{self.request_limit}")
            
        if self.tokens_per_request_limit and estimated_tokens > self.tokens_per_request_limit:
            raise Exception(f"Limite de tokens por requisição excedido: {estimated_tokens}/{self.tokens_per_request_limit}")
            
        if self.tokens_per_minute_limit and self.minute_tokens >= self.tokens_per_minute_limit:
            raise Exception(f"Limite de tokens por minuto excedido: {self.minute_tokens}/{self.tokens_per_minute_limit}")
            
        return True
    
    def record_usage(self, prompt: str, response: str) -> Dict[str, Any]:
        # Registro simplificado - em uma implementação real, você usaria os dados de uso reais
        # retornados pela API do modelo
        prompt_tokens = len(prompt.split()) * 1.3  # Estimativa simples
        response_tokens = len(response.split()) * 1.3  # Estimativa simples
        total = prompt_tokens + response_tokens
        
        # Atualizar contadores
        self.request_count += 1
        self.total_tokens += total
        self.minute_tokens += total
        
        # Registrar uso
        usage_entry = {
            "prompt_tokens": int(prompt_tokens),
            "response_tokens": int(response_tokens),
            "total_tokens": int(total),
            "timestamp": time.time()
        }
        
        self.usage_log[f"request_{self.request_count}"] = usage_entry
        return usage_entry

# Demonstração do monitoramento de uso
async def demonstrar_monitoramento_uso():
    print("🔍 Configurando monitoramento de uso para o agente...")
    
    # Criar monitor de uso
    monitor = UsageMonitor(
        request_limit=10,
        tokens_per_request_limit=4000,
        tokens_per_minute_limit=20000
    )
    
    # Criar agente
    agent = Agent(
        'groq:openai/gpt-oss-120b',
        system_prompt="Você é um assistente conciso e eficiente."
    )
    
    try:
        # Verificar limites antes da requisição
        prompt = "Explique brevemente o que é IA Generativa"
        monitor.check_limits(estimated_tokens=len(prompt.split()) * 2)  # Estimativa simples
        
        # Executar uma consulta
        print("📝 Executando consulta com monitoramento de uso...")
        result = await agent.run(prompt)
        
        # Registrar uso após a requisição
        usage = monitor.record_usage(prompt, result.output)
        
        # Mostrar resultados
        print("\n✅ Consulta bem-sucedida!")
        print(f"📊 Resposta: {result.output[:100]}...")
        print(f"📈 Uso estimado: {usage}")
        print(f"📊 Total de requisições: {monitor.request_count}")
        print(f"📊 Total de tokens: {monitor.total_tokens}")
        
        return result, monitor
    
    except Exception as e:
        print(f"❌ Erro: {e}")
        return None, monitor

# Executar a demonstração
await demonstrar_monitoramento_uso()

🔍 Configurando monitoramento de uso para o agente...
📝 Executando consulta com monitoramento de uso...

✅ Consulta bem-sucedida!
📊 Resposta: IA Generativa é um ramo da inteligência artificial que cria conteúdo novo — como texto, imagens, áud...
📈 Uso estimado: {'prompt_tokens': 9, 'response_tokens': 93, 'total_tokens': 102, 'timestamp': 1793808977.1092055}
📊 Total de requisições: 1
📊 Total de tokens: 102.7


(AgentRunResult(output='IA Generativa é um ramo da inteligência artificial que cria conteúdo novo — como texto, imagens, áudio ou código — a partir de padrões aprendidos em grandes volumes de dados. Modelos como GPT (para texto) ou DALL·E (para imagens) recebem um prompt (instrução) e, usando técnicas de aprendizado profundo (ex.: redes neurais transformadoras), geram respostas que imitam ou combinam elementos do material de treinamento, permitindo a produção automática e criativa de informações.'),
 <__main__.UsageMonitor at 0x186d028f470>)